In [ ]:
!pip install --upgrade nbformat
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import matplotlib.dates as mdates
from statsmodels.tsa.stattools import adfuller
import os
import plotly.express as px


DATA_DIR = "data/sentiment"
BTC_FILE = os.path.join(DATA_DIR, "bitcoin_with_vader_normalized.csv")
STOCKS_FILE = os.path.join(DATA_DIR, "stock_with_vader_normalized.csv")

NEG_THRESHOLD = -0.25
POS_THRESHOLD = 0.25


TICKERS_OF_INTEREST = ['TSLA', 'AMZN', 'BTC', 'AAPL', 'MSFT', 'GOOG']

ADF_ALPHA = 0.05


print("Libraries imported and configuration set.")
print(f"Bitcoin data path: {BTC_FILE}")
print(f"Stocks data path: {STOCKS_FILE}")
print(f"Tickers of interest: {TICKERS_OF_INTEREST}")

Libraries imported and configuration set.
Bitcoin data path: data/sentiment/bitcoin_with_vader_normalized.csv
Stocks data path: data/sentiment/stock_with_vader_normalized.csv
Tickers of interest: ['TSLA', 'AMZN', 'BTC', 'AAPL', 'MSFT', 'GOOG']


Cell 2: Load and Combine Data

In [ ]:

try:
    df_btc = pd.read_csv(BTC_FILE)
    print(f"Loaded {len(df_btc)} records from {BTC_FILE}")
except FileNotFoundError:
    print(f"ERROR: File not found at {BTC_FILE}")
    df_btc = pd.DataFrame()

try:
    df_stocks = pd.read_csv(STOCKS_FILE)
    print(f"Loaded {len(df_stocks)} records from {STOCKS_FILE}")
except FileNotFoundError:
    print(f"ERROR: File not found at {STOCKS_FILE}")
    df_stocks = pd.DataFrame()


df_combined = pd.concat([df_btc, df_stocks], ignore_index=True)
print(f"Combined dataframe shape: {df_combined.shape}")
# --- End Load Data ---


df_combined['datetime'] = pd.to_datetime(df_combined['datetime'], errors='coerce', utc=True)
df_combined.dropna(subset=['datetime'], inplace=True)
df_combined['date'] = df_combined['datetime'].dt.date
df_combined['date'] = pd.to_datetime(df_combined['date'])

print("\nPreprocessing steps:")
print("Converted 'datetime' to datetime objects.")
print("Extracted 'date' column.")
print(f"Shape after preprocessing: {df_combined.shape}")
df_combined.head()


Loaded 13816953 records from data/sentiment/bitcoin_with_vader_normalized.csv
Loaded 80793 records from data/sentiment/stock_with_vader_normalized.csv
Combined dataframe shape: (13897746, 6)

Preprocessing steps:
Converted 'datetime' to datetime objects.
Extracted 'date' column.
Shape after preprocessing: (13897746, 7)


,datetime,ticker,neg,neu,pos,compound,date
0,2021-06-30 17:42:34+00:00,BTC,0.203,0.797,0.000,-0.6523,2021-06-30
1,2021-06-30 17:42:41+00:00,BTC,0.107,0.741,0.152,0.2023,2021-06-30
2,2021-06-30 17:42:48+00:00,BTC,0.063,0.798,0.139,0.4215,2021-06-30
3,2021-06-30 17:42:52+00:00,BTC,0.000,0.871,0.129,0.6369,2021-06-30
4,2021-06-30 17:42:54+00:00,BTC,0.000,1.000,0.000,0.0000,2021-06-30


Cell 3: Sentiment Classification

In [ ]:

conditions = [
    df_combined['compound'] < NEG_THRESHOLD,
    df_combined['compound'] > POS_THRESHOLD
]

choices = ['NEGATIVE', 'POSITIVE']


df_combined['sentiment_category'] = np.select(conditions, choices, default='NEUTRAL')

print("\nSentiment classification based on 'compound' score:")
print(f"Negative threshold: < {NEG_THRESHOLD}")
print(f"Positive threshold: > {POS_THRESHOLD}")
print("Added 'sentiment_category' column.")

df_combined[['ticker', 'compound', 'sentiment_category']].head()



Sentiment classification based on 'compound' score:
Negative threshold: < -0.25
Positive threshold: > 0.25
Added 'sentiment_category' column.


,ticker,compound,sentiment_category
0,BTC,-0.6523,NEGATIVE
1,BTC,0.2023,NEUTRAL
2,BTC,0.4215,POSITIVE
3,BTC,0.6369,POSITIVE
4,BTC,0.0000,NEUTRAL


In [ ]:

print("TABLE: Total Tweet Counts per Ticker\n")


df_total_counts = df_combined[df_combined['ticker'].isin(TICKERS_OF_INTEREST)].copy()

if not df_total_counts.empty:

    total_counts = df_total_counts.groupby('ticker').size()


    total_counts_table = pd.DataFrame(total_counts, columns=['Total Tweets'])
    total_counts_table.index.name = 'Ticker'

    display(total_counts_table)
else:
    print(f"No data found for the specified tickers: {TICKERS_OF_INTEREST}")


TABLE: Total Tweet Counts per Ticker



,Total Tweets
Ticker,
AAPL,5056
AMZN,4089
BTC,13816953
GOOG,1291
MSFT,4089
TSLA,37422


Cell 4: TABULKA: SUM sentiment po aktívach

In [ ]:

print("\nTABLE: Sentiment Breakdown per Ticker (Count and Percentage)")
print(f"Compound Thresholds: Negative < {NEG_THRESHOLD}, Positive > {POS_THRESHOLD}\n")


df_table_data = df_combined[df_combined['ticker'].isin(TICKERS_OF_INTEREST)].copy()

if not df_table_data.empty:

    sentiment_counts = df_table_data.groupby(['ticker', 'sentiment_category']).size()

    summary_table = sentiment_counts.unstack(fill_value=0)

    cols_order = ['POSITIVE', 'NEUTRAL', 'NEGATIVE']
    for col in cols_order:
        if col not in summary_table.columns:
            summary_table[col] = 0
    summary_table = summary_table[cols_order]


    summary_table.index.name = 'Ticker'

    row_totals = summary_table.sum(axis=1)

    summary_table_percent = summary_table.copy()

    def format_with_percent(count, total):
        if total == 0: 
            return f"{count} (NaN%)"
        percent = (count / total) * 100
        
        return f"{count} ({percent:.1f}%)"


    for col in summary_table_percent.columns:
        summary_table_percent[col] = summary_table_percent.apply(
            lambda row: format_with_percent(row[col], row_totals.loc[row.name]), axis=1
        )


    display(summary_table_percent)

else:
    print(f"No sentiment data found for the specified tickers: {TICKERS_OF_INTEREST}")



TABLE: Sentiment Breakdown per Ticker (Count and Percentage)
Compound Thresholds: Negative < -0.25, Positive > 0.25



sentiment_category,POSITIVE,NEUTRAL,NEGATIVE
Ticker,,,
AAPL,1978 (39.1%),2233 (44.2%),845 (16.7%)
AMZN,1850 (45.2%),1607 (39.3%),632 (15.5%)
BTC,6001856 (43.4%),5434692 (39.3%),2380405 (17.2%)
GOOG,604 (46.8%),507 (39.3%),180 (13.9%)
MSFT,1850 (45.2%),1607 (39.3%),632 (15.5%)
TSLA,16626 (44.4%),14357 (38.4%),6439 (17.2%)


Cell 5: Calculate Daily Average Sentiment

In [ ]:

daily_sentiment = df_combined.groupby(['date', 'ticker'])['compound'].mean()

daily_sentiment_pivot = daily_sentiment.unstack(level='ticker')

available_tickers = [t for t in TICKERS_OF_INTEREST if t in daily_sentiment_pivot.columns]

if not daily_sentiment_pivot.empty and available_tickers:
    daily_sentiment_pivot = daily_sentiment_pivot[available_tickers] 
    print("\nCalculated daily average sentiment (S_t) per ticker:")
    display(daily_sentiment_pivot.head())


else:
     print("Could not calculate daily sentiment. Initial data might be empty or filtering failed.")



Calculated daily average sentiment (S_t) per ticker:


ticker,TSLA,AMZN,BTC,AAPL,MSFT,GOOG
date,,,,,,
2021-01-01,NaN,NaN,0.213080,NaN,NaN,NaN
2021-01-02,NaN,NaN,0.149149,NaN,NaN,NaN
2021-01-03,NaN,NaN,0.186784,NaN,NaN,NaN
2021-01-04,NaN,NaN,0.171129,NaN,NaN,NaN
2021-01-05,NaN,NaN,0.183922,NaN,NaN,NaN


Cell 6: GRAF: Vývoj denného sentimentu v čase

In [ ]:

print("\nGRAPH: Development of daily sentiment over time by asset type (Interactive)")

if 'daily_sentiment_pivot' in locals() and not daily_sentiment_pivot.empty:

    if not daily_sentiment_pivot.isnull().all().all(): 


        fig = px.line(daily_sentiment_pivot,
                      x=daily_sentiment_pivot.index,
                      y=daily_sentiment_pivot.columns, 
                      labels={'value': 'Average Daily Sentiment Score (S_t)', 
                              'date': 'Date',                  
                              'variable': 'Ticker'},           
                      title='Development of Daily Average Sentiment (Compound Score) Over Time')


        fig.update_traces(mode='lines+markers', marker=dict(size=4))


        fig.update_layout(
            xaxis_title='Date',
            yaxis_title='Average Daily Sentiment Score (S_t)',
            legend_title='Ticker',
            hovermode='x unified' 
        )

        fig.show()

    else:
        print("Cannot plot: All data in daily_sentiment_pivot is NaN.")

else:
    print("Cannot plot: Daily sentiment data ('daily_sentiment_pivot') is not available or empty.")



GRAPH: Development of daily sentiment over time by asset type (Interactive)


Cell 7: Stacionarita: ADF-test

In [ ]:

print("\nStationarity: ADF-Test for Daily Sentiment Time Series (S_t)")
print(f"Significance level (alpha) = {ADF_ALPHA}\n")

if 'daily_sentiment_pivot' in locals() and not daily_sentiment_pivot.empty:
    for ticker in daily_sentiment_pivot.columns:
        print(f"--- ADF Test for {ticker} Daily Sentiment (S_t) ---")

        time_series = daily_sentiment_pivot[ticker].dropna()

        if len(time_series) < 10: 
             print(f"Skipping {ticker}: Not enough non-NaN data points ({len(time_series)}) for ADF test.")
             print("-" * (len(ticker) + 15)) 
             continue

        try:
        
            adf_result = adfuller(time_series)

            adf_statistic = adf_result[0]
            p_value = adf_result[1]
            critical_values = adf_result[4]

            print(f"ADF Statistic: {adf_statistic:.4f}")
            print(f"P-value: {p_value:.4f}")
            print("Critical Values:")
            for key, value in critical_values.items():
                print(f"\t{key}: {value:.4f}")

           
            if p_value <= ADF_ALPHA:
                print(f"Result: The series is likely Stationary (Reject H0) at alpha={ADF_ALPHA}")
            else:
                print(f"Result: The series is likely Non-Stationary (Fail to reject H0) at alpha={ADF_ALPHA}")
                print("Suggestion: Consider using the differenced series (ΔS_t = S_t - S_{t-1}) for modeling.")

        except Exception as e:
            print(f"Could not perform ADF test for {ticker}. Error: {e}")

        print("-" * (len(ticker) + 15)) 
else:
     print("Cannot perform ADF test: Daily sentiment data ('daily_sentiment_pivot') is not available or empty.")



Stationarity: ADF-Test for Daily Sentiment Time Series (S_t)
Significance level (alpha) = 0.05

--- ADF Test for TSLA Daily Sentiment (S_t) ---
ADF Statistic: -14.9452
P-value: 0.0000
Critical Values:
	1%: -3.4484
	5%: -2.8695
	10%: -2.5710
Result: The series is likely Stationary (Reject H0) at alpha=0.05
-------------------
--- ADF Test for AMZN Daily Sentiment (S_t) ---
ADF Statistic: -9.7124
P-value: 0.0000
Critical Values:
	1%: -3.4485
	5%: -2.8696
	10%: -2.5710
Result: The series is likely Stationary (Reject H0) at alpha=0.05
-------------------
--- ADF Test for BTC Daily Sentiment (S_t) ---
ADF Statistic: -3.1083
P-value: 0.0259
Critical Values:
	1%: -3.4425
	5%: -2.8669
	10%: -2.5696
Result: The series is likely Stationary (Reject H0) at alpha=0.05
------------------
--- ADF Test for AAPL Daily Sentiment (S_t) ---
ADF Statistic: -10.4864
P-value: 0.0000
Critical Values:
	1%: -3.4485
	5%: -2.8695
	10%: -2.5710
Result: The series is likely Stationary (Reject H0) at alpha=0.05
---